In [1]:
!ls /kaggle/input/elastic-breakhis/elastic_breakhis

adenosis	  fibroadenoma	     mucinous_carcinoma   phyllodes_tumor
ductal_carcinoma  lobular_carcinoma  papillary_carcinoma  tubular_adenoma


In [2]:
import os
import shutil
import random
from collections import defaultdict


CLASS_MAPPING = {
    'SOB_B_A': 'adenosis',
    'SOB_B_F': 'fibroadenoma',
    'SOB_B_PT': 'phyllodes_tumor',
    'SOB_B_TA': 'tubular_adenoma',
    'SOB_M_DC': 'ductal_carcinoma',
    'SOB_M_LC': 'lobular_carcinoma',
    'SOB_M_MC': 'mucinous_carcinoma',
    'SOB_M_PC': 'papillary_carcinoma'
}


REVERSE_MAPPING = {v: k for k, v in CLASS_MAPPING.items()}


In [3]:
breakhis_train_path = '/kaggle/input/split-data/BreakHis_splits/train'  
kaggle_dataset1_path = '/kaggle/input/gamma-breakhis'     
kaggle_dataset2_path = '/kaggle/input/elastic-breakhis/elastic_breakhis'       
output_path = '/kaggle/working/final1'     


TARGET_COUNT = 1000
DATASET1_RATIO = 0.6  
DATASET2_RATIO = 0.4  


os.makedirs(output_path, exist_ok=True)
for breakhis_code in CLASS_MAPPING.keys():
    os.makedirs(os.path.join(output_path, breakhis_code), exist_ok=True)

In [4]:

breakhis_counts = defaultdict(int)
for breakhis_code in CLASS_MAPPING.keys():
    subclass_path = os.path.join(breakhis_train_path, breakhis_code)
    if os.path.exists(subclass_path):
        breakhis_counts[breakhis_code] = len(os.listdir(subclass_path))

print("Current counts per class in BreakHis:")
for breakhis_code, count in breakhis_counts.items():
    print(f"{CLASS_MAPPING[breakhis_code]} ({breakhis_code}): {count} images")
print(f"\nTarget count per class: {TARGET_COUNT}")


kaggle_dataset1_images = defaultdict(list)
kaggle_dataset2_images = defaultdict(list)

Current counts per class in BreakHis:
adenosis (SOB_B_A): 294 images
fibroadenoma (SOB_B_F): 864 images
phyllodes_tumor (SOB_B_PT): 303 images
tubular_adenoma (SOB_B_TA): 419 images
ductal_carcinoma (SOB_M_DC): 3301 images
lobular_carcinoma (SOB_M_LC): 476 images
mucinous_carcinoma (SOB_M_MC): 642 images
papillary_carcinoma (SOB_M_PC): 410 images

Target count per class: 1000


In [5]:
def scan_kaggle_dataset(dataset_path, storage_dict):
    for root, dirs, files in os.walk(dataset_path):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
                
                dir_name = os.path.basename(root).lower()
                
                for class_name, breakhis_code in CLASS_MAPPING.items():
                    if class_name.lower().replace('_', ' ') in dir_name.lower() or \
                       CLASS_MAPPING[class_name].lower() in dir_name.lower():
                        storage_dict[class_name].append(os.path.join(root, file))
                        break


scan_kaggle_dataset(kaggle_dataset1_path, kaggle_dataset1_images)
scan_kaggle_dataset(kaggle_dataset2_path, kaggle_dataset2_images)

print("\nAvailable additional images from Kaggle Dataset 1:")
for breakhis_code, images in kaggle_dataset1_images.items():
    print(f"{CLASS_MAPPING[breakhis_code]} ({breakhis_code}): {len(images)} images")
    
print("\nAvailable additional images from Kaggle Dataset 2:")
for breakhis_code, images in kaggle_dataset2_images.items():
    print(f"{CLASS_MAPPING[breakhis_code]} ({breakhis_code}): {len(images)} images")


Available additional images from Kaggle Dataset 1:
adenosis (SOB_B_A): 444 images
lobular_carcinoma (SOB_M_LC): 626 images
phyllodes_tumor (SOB_B_PT): 453 images
mucinous_carcinoma (SOB_M_MC): 792 images
ductal_carcinoma (SOB_M_DC): 3451 images
papillary_carcinoma (SOB_M_PC): 560 images
tubular_adenoma (SOB_B_TA): 569 images
fibroadenoma (SOB_B_F): 1014 images

Available additional images from Kaggle Dataset 2:
adenosis (SOB_B_A): 444 images
lobular_carcinoma (SOB_M_LC): 626 images
phyllodes_tumor (SOB_B_PT): 453 images
mucinous_carcinoma (SOB_M_MC): 792 images
ductal_carcinoma (SOB_M_DC): 3451 images
papillary_carcinoma (SOB_M_PC): 560 images
tubular_adenoma (SOB_B_TA): 569 images
fibroadenoma (SOB_B_F): 1014 images


In [6]:
for breakhis_code in CLASS_MAPPING.keys():
    src_dir = os.path.join(breakhis_train_path, breakhis_code)
    dst_dir = os.path.join(output_path, breakhis_code)
    
    current_count = breakhis_counts.get(breakhis_code, 0)
    
    
    if current_count >= TARGET_COUNT:
        all_images = os.listdir(src_dir)
        selected = random.sample(all_images, TARGET_COUNT)
        for file in selected:
            shutil.copy2(os.path.join(src_dir, file), os.path.join(dst_dir, file))
        print(f"{CLASS_MAPPING[breakhis_code]} ({breakhis_code}): Had enough ({current_count}), randomly selected {TARGET_COUNT}")
        continue
    
    
    if os.path.exists(src_dir):
        for file in os.listdir(src_dir):
            shutil.copy2(os.path.join(src_dir, file), os.path.join(dst_dir, file))
    
    needed = TARGET_COUNT - current_count
    needed_from_dataset1 = int(needed * DATASET1_RATIO)
    needed_from_dataset2 = needed - needed_from_dataset1
    
    available_dataset1 = len(kaggle_dataset1_images.get(breakhis_code, []))
    available_dataset2 = len(kaggle_dataset2_images.get(breakhis_code, []))
    
    if available_dataset1 < needed_from_dataset1:
        deficit = needed_from_dataset1 - available_dataset1
        needed_from_dataset1 = available_dataset1
        needed_from_dataset2 += deficit
    
    if available_dataset2 < needed_from_dataset2:
        deficit = needed_from_dataset2 - available_dataset2
        needed_from_dataset2 = available_dataset2
        needed_from_dataset1 += deficit
   
    total_available = available_dataset1 + available_dataset2
    if total_available < needed:
        print(f"Warning: Not enough additional images for {CLASS_MAPPING[breakhis_code]}. Needed {needed}, only {total_available} available.")
        needed = total_available
        
        needed_from_dataset1 = int(needed * DATASET1_RATIO)
        needed_from_dataset2 = needed - needed_from_dataset1
    
    
    if needed_from_dataset1 > 0 and breakhis_code in kaggle_dataset1_images:
        selected_dataset1 = random.sample(kaggle_dataset1_images[breakhis_code], needed_from_dataset1)
        for i, src_path in enumerate(selected_dataset1):
            ext = os.path.splitext(src_path)[1]
            dst_path = os.path.join(dst_dir, f"aug1_{i}{ext}")
            shutil.copy2(src_path, dst_path)
    
    if needed_from_dataset2 > 0 and breakhis_code in kaggle_dataset2_images:
        selected_dataset2 = random.sample(kaggle_dataset2_images[breakhis_code], needed_from_dataset2)
        for i, src_path in enumerate(selected_dataset2):
            ext = os.path.splitext(src_path)[1]
            dst_path = os.path.join(dst_dir, f"aug2_{i}{ext}")
            shutil.copy2(src_path, dst_path)
    
    final_count = current_count + needed_from_dataset1 + needed_from_dataset2
    print(f"{CLASS_MAPPING[breakhis_code]} ({breakhis_code}): Original {current_count}, added {needed_from_dataset1 + needed_from_dataset2} ({needed_from_dataset1} from DS1, {needed_from_dataset2} from DS2), total {final_count}")

adenosis (SOB_B_A): Original 294, added 706 (423 from DS1, 283 from DS2), total 1000
fibroadenoma (SOB_B_F): Original 864, added 136 (81 from DS1, 55 from DS2), total 1000
phyllodes_tumor (SOB_B_PT): Original 303, added 697 (418 from DS1, 279 from DS2), total 1000
tubular_adenoma (SOB_B_TA): Original 419, added 581 (348 from DS1, 233 from DS2), total 1000
ductal_carcinoma (SOB_M_DC): Had enough (3301), randomly selected 1000
lobular_carcinoma (SOB_M_LC): Original 476, added 524 (314 from DS1, 210 from DS2), total 1000
mucinous_carcinoma (SOB_M_MC): Original 642, added 358 (214 from DS1, 144 from DS2), total 1000
papillary_carcinoma (SOB_M_PC): Original 410, added 590 (354 from DS1, 236 from DS2), total 1000


In [7]:
print("\nFinal counts per class:")
for breakhis_code in CLASS_MAPPING.keys():
    count = len(os.listdir(os.path.join(output_path, breakhis_code))) if os.path.exists(os.path.join(output_path, breakhis_code)) else 0
    print(f"{CLASS_MAPPING[breakhis_code]} ({breakhis_code}): {count} images")

shutil.make_archive('/kaggle/working/balanced_dataset', 'zip', output_path)
print("\nDataset created and compressed as balanced_dataset.zip")



Final counts per class:
adenosis (SOB_B_A): 1000 images
fibroadenoma (SOB_B_F): 1000 images
phyllodes_tumor (SOB_B_PT): 1000 images
tubular_adenoma (SOB_B_TA): 1000 images
ductal_carcinoma (SOB_M_DC): 1000 images
lobular_carcinoma (SOB_M_LC): 1000 images
mucinous_carcinoma (SOB_M_MC): 1000 images
papillary_carcinoma (SOB_M_PC): 1000 images

Dataset created and compressed as balanced_dataset.zip
